In [2]:
import uproot
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.lines import Line2D



In [3]:
# Open ROOT file and tree
file = uproot.open("e20gev.root")
tree = file["tree"]
print(file.keys())
print(tree.keys())


['tree;112', 'tree;111']
['step_preX', 'step_preY', 'step_preZ', 'step_postX', 'step_postY', 'step_postZ', 'step_kinE', 'step_edep', 'step_trackID', 'step_parentID', 'step_PDG', 'E', 'x', 'y', 'z', 'costh', 'vertexX', 'vertexY', 'vertexZ', 'vertexT', 'finalE', 'finalX', 'finalY', 'finalZ', 'px', 'py', 'pz', 'finalPx', 'finalPy', 'finalPz', 'finalCosth', 'theta', 'phi', 'finalPhi', 'finalPhiDeg', 'totalEdep', 'nSteps', 'nSecondaries', 'interactionType', 'secNames', 'secEnergies', 'nGamma', 'nElectron', 'nPositron', 'nProtonSec', 'nNeutron', 'nPionPlus', 'nPionMinus', 'nPionZero', 'nMuonPlus', 'nMuonMinus', 'nTauPlus', 'nTauMinus', 'nKaonPlus', 'nKaonMinus', 'nKaonZero', 'nKaonZeroL', 'nKaonZeroS', 'secTotalE', 'secMeanE', 'secTrackLength', 'secStartX', 'secStartY', 'secStartZ', 'secEndX', 'secEndY', 'secEndZ', 'nBackward', 'nDecay', 'nCompton', 'nPairProd', 'nIonisation', 'nBremsstrahlung', 'nnPhotoElectric', 'nAnnihilation', 'targetZ', 'targetA', 'targetPDG']


In [4]:
n_events = len(tree["step_trackID"].array())

for i in range(n_events):

    # trackID and parentID for this event
    trackID  = np.array(tree["step_trackID"].array()[i].to_list())
    parentID = np.array(tree["step_parentID"].array()[i].to_list())
    PDG      = np.array(tree["step_PDG"].array()[i].to_list())

    # Find secondary particles (parentID != 0)
    secondary_mask = (parentID != 0)
    nSecondaries = np.sum(secondary_mask)

    # Count e-, e+, gamma among secondaries
    n_electron = np.sum((PDG[secondary_mask] == 11))
    n_positron = np.sum((PDG[secondary_mask] == -11))
    n_gamma    = np.sum((PDG[secondary_mask] == 22))

    # Print event if any secondary exists or if e+/e-/gamma exist
    #if nSecondaries > 0 or n_electron > 0 or n_positron > 0 or n_gamma > 0:
     #   print(f"Event {i}: nSecondaries={nSecondaries}, "
      #        f"e⁻={n_electron}, e⁺={n_positron}, γ={n_gamma}")

In [ ]:
# Choose an event
event_idx = 1

MAX_GENERATION = np.inf      # Option 2
MAX_STEPS = np.inf         # Option 3
ZOOM_VERTEX = False      # Option 4
ZOOM_SIZE = 400         # mm around vertex


trackID  = np.array(tree["step_trackID"].array()[event_idx].to_list())
parentID = np.array(tree["step_parentID"].array()[event_idx].to_list())
PDG      = np.array(tree["step_PDG"].array()[event_idx].to_list())

preX = np.array(tree["step_preX"].array()[event_idx].to_list())
preY = np.array(tree["step_preY"].array()[event_idx].to_list())
preZ = np.array(tree["step_preZ"].array()[event_idx].to_list())

postX = np.array(tree["step_postX"].array()[event_idx].to_list())
postY = np.array(tree["step_postY"].array()[event_idx].to_list())
postZ = np.array(tree["step_postZ"].array()[event_idx].to_list())

vx = tree["vertexX"].array()[event_idx]
vy = tree["vertexY"].array()[event_idx]
vz = tree["vertexZ"].array()[event_idx]



In [ ]:
pdg_map = {

     11: ("e⁻", "red"),
    -11: ("e⁺", "green"),

     13: ("μ⁻", "blue"),
    -13: ("μ⁺", "cyan"),

     22: ("γ", "orange"),

    211: ("π⁺", "purple"),
   -211: ("π⁻", "pink"),

    2212: ("p", "brown"),
    2112: ("n", "gray"),

    321: ("K⁺", "darkred"),
   -321: ("K⁻", "lightcoral"),

     15: ("τ⁻", "navy"),
    -15: ("τ⁺", "skyblue"),

    # --- neutrinos (distinct colors) ---
     12: ("νₑ", "gold"),
    -12: ("ν̄ₑ", "khaki"),

     14: ("ν_μ", "darkgreen"),
    -14: ("ν̄_μ", "limegreen"),

     16: ("ν_τ", "magenta"),
    -16: ("ν̄_τ", "violet"),
}
unique_tracks = np.unique(trackID)

# 3D display

In [ ]:

# -------------------------------
# 3D Figure
# -------------------------------
fig = plt.figure(figsize=(12,9))
ax = fig.add_subplot(111, projection='3d')

# -------------------------------
# Plot each track
# -------------------------------
for tid in unique_tracks:

    mask = (trackID == tid)

    x0 = preX[mask]
    y0 = preY[mask] if 'preY' in locals() else np.zeros_like(x0)  # fallback if Y missing
    z0 = preZ[mask]

    x1 = postX[mask]
    y1 = postY[mask] if 'postY' in locals() else np.zeros_like(x1)
    z1 = postZ[mask]

    pdg = PDG[mask][0]
    name, color = pdg_map.get(pdg, (f"PDG {pdg}", "black"))

    gen = generations[tid]
    linewidth = max(0.6, 3 - gen)  # thicker for early generations

    for a,b,c,d,e,f in zip(x0, y0, z0, x1, y1, z1):
        ax.plot(
            [a,c], [b,d], [e,f],
            color=color,
            linewidth=linewidth,
            alpha=0.8
        )

# -------------------------------
# Plot vertex
# -------------------------------
ax.scatter(vx, vy, vz, color='black', s=80, label='Vertex', zorder=5)

# -------------------------------
# Labels
# -------------------------------
ax.set_xlabel('X [mm]')
ax.set_ylabel('Y [mm]')
ax.set_zlabel('Z [mm] (beam direction)')
ax.set_title(f'3D Event Display — Event {event_idx}')

# -------------------------------
# Build legend
# -------------------------------
legend_elements = []

for pdg, (name, color) in pdg_map.items():
    legend_elements.append(Line2D([0],[0], color=color, lw=2, label=name))

legend_elements.append(Line2D([0],[0], marker='o', color='black', label='Vertex',
                              markersize=8, linestyle='None'))

ax.legend(handles=legend_elements, loc='upper right', fontsize=9)

plt.show()

# 2D display

In [ ]:
track_parent_map = {}
for tid in unique_tracks:

    mask = (trackID == tid)

    parent = parentID[mask][0]

    track_parent_map[tid] = parent

# -------------------------------
# Determine generation depth
# -------------------------------

generations = {}

for tid in unique_tracks:

    parent = track_parent_map[tid]

    if parent == 0:

        generations[tid] = 0

    else:

        generations[tid] = (
            generations.get(parent, 0) + 1
        )

# -------------------------------
# Plot
# -------------------------------

plt.figure(figsize=(12,7))

for tid in unique_tracks:

    mask = (trackID == tid)

    x0 = preX[mask]
    z0 = preZ[mask]

    x1 = postX[mask]
    z1 = postZ[mask]

    pdg = PDG[mask][0]

    name, color = pdg_map.get(
        pdg,
        (f"PDG {pdg}", "black")
    )

    gen = generations[tid]

    # thicker lines for earlier generations
    linewidth = max(0.6, 3 - gen)

    for a,b,c,d in zip(z0,x0,z1,x1):

        plt.plot(
            [a,c],
            [b,d],
            color=color,
            linewidth=linewidth,
            alpha=0.8
        )

# -------------------------------
# Plot interaction vertex
# -------------------------------

plt.scatter(
    vz,
    vx,
    color="black",
    s=80,
    label="Vertex",
    zorder=5
)

# -------------------------------
# Build legend
# -------------------------------

legend_elements = []

for pdg, (name, color) in pdg_map.items():

    legend_elements.append(

        Line2D(
            [0],
            [0],
            color=color,
            lw=2,
            label=name
        )
    )

legend_elements.append(

    Line2D(
        [0],
        [0],
        marker='o',
        color='black',
        label='Vertex',
        markersize=8,
        linestyle='None'
    )
)

plt.legend(
    handles=legend_elements,
    loc="upper right",
    fontsize=9
)

# -------------------------------
# Axis labels
# -------------------------------

plt.xlabel("Z [mm]")
plt.ylabel("X [mm]")

plt.title(
    f"Event Display — Event {event_idx}"
)

plt.grid(True)

plt.tight_layout()

plt.show()

# limit generation, steps, and group by track ID

In [ ]:
# Build parent map

track_parent_map = {}

for tid in unique_tracks:

    mask = (trackID == tid)

    parent = parentID[mask][0]

    track_parent_map[tid] = parent


# ============================================
# OPTION 2:
# Compute generation depth
# ============================================

generations = {}

def get_generation(tid):

    parent = track_parent_map.get(tid, 0)

    if parent == 0:
        return 0

    if parent in generations:
        return generations[parent] + 1

    return 1


for tid in unique_tracks:

    generations[tid] = get_generation(tid)


# ============================================
# Plot
# ============================================

plt.figure(figsize=(12,7))

step_counter = 0

for tid in unique_tracks:

    gen = generations.get(tid, 0)

    # Skip high generations
    if gen > MAX_GENERATION:
        continue

    mask = (trackID == tid)

    x0 = preX[mask]
    z0 = preZ[mask]

    x1 = postX[mask]
    z1 = postZ[mask]

    pdg = PDG[mask][0]

    name, color = pdg_map.get(
        pdg,
        (f"PDG {pdg}", "black")
    )

    # thicker for early generations
    linewidth = max(0.6, 3 - gen)

    # ========================================
    # OPTION 3:
    # Limit number of steps
    # ========================================

    for a,b,c,d in zip(z0,x0,z1,x1):

        if step_counter > MAX_STEPS:
            break

        plt.plot(
            [a,c],
            [b,d],
            color=color,
            linewidth=linewidth,
            alpha=0.8
        )

        step_counter += 1


# ============================================
# Plot vertex
# ============================================

plt.scatter(
    vz,
    vx,
    color="black",
    s=80,
    label="Vertex",
    zorder=5
)


# ============================================
# Build legend
# ============================================

legend_elements = []

used_pdgs = np.unique(PDG)

for pdg in used_pdgs:

    if pdg in pdg_map:

        name, color = pdg_map[pdg]

        legend_elements.append(
            Line2D(
                [0],
                [0],
                color=color,
                lw=2,
                label=name
            )
        )

legend_elements.append(

    Line2D(
        [0],
        [0],
        marker='o',
        color='black',
        label='Vertex',
        markersize=8,
        linestyle='None'
    )
)

plt.legend(
    handles=legend_elements,
    loc="upper right",
    fontsize=9
)


# ============================================
# OPTION 4:
# Zoom near vertex
# ============================================

if ZOOM_VERTEX:

    plt.xlim(
        vz - ZOOM_SIZE,
        vz + ZOOM_SIZE
    )

    plt.ylim(
        vx - ZOOM_SIZE,
        vx + ZOOM_SIZE
    )


# ============================================
# Labels
# ============================================

plt.xlabel("Z [mm] (beam direction)")
plt.ylabel("X [mm]")

plt.title(
    f"Event Display — Event {event_idx}"
)

plt.grid(True)

plt.tight_layout()

plt.show()

In [ ]:
vx = tree["vertexX"].array()
vy = tree["vertexY"].array()
vz = tree["vertexZ"].array()
vt = tree["vertexT"].array() 


import matplotlib.pyplot as plt

# Histogram for each vertex coordinate
plt.figure(figsize=(12,8))

plt.subplot(2,2,1)
plt.hist(vx, bins=50, color='red', alpha=0.7)
plt.xlabel('Vertex X [mm]')
plt.ylabel('Counts')
plt.grid(True)

plt.subplot(2,2,2)
plt.hist(vy, bins=50, color='green', alpha=0.7)
plt.xlabel('Vertex Y [mm]')
plt.ylabel('Counts')
plt.grid(True)

plt.subplot(2,2,3)
plt.hist(vz, bins=50, color='blue', alpha=0.7)
plt.xlabel('Vertex Z [mm]')
plt.ylabel('Counts')
plt.grid(True)

plt.subplot(2,2,4)
plt.hist(vt, bins=50, color='purple', alpha=0.7)
plt.xlabel('Vertex T [ns]')
plt.ylabel('Counts')
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
trackID  = tree["step_trackID"].array()[event_idx].to_list()
parentID = tree["step_parentID"].array()[event_idx].to_list()
PDG      = tree["step_PDG"].array()[event_idx].to_list()

plt.figure()
plt.hist(trackID, bins=np.max(trackID)-np.min(trackID)+1, color='skyblue', alpha=0.8)
plt.xlabel('Track ID')
plt.ylabel('Counts')
plt.title(f'Event {event_idx} Track IDs')
plt.grid(True)
plt.show()

plt.figure()
plt.hist(parentID, bins=np.max(parentID)-np.min(parentID)+1, color='orange', alpha=0.8)
plt.xlabel('Parent ID')
plt.ylabel('Counts')
plt.title(f'Event {event_idx} Parent IDs')
plt.grid(True)
plt.show()

# Track ID vs Parent ID
plt.scatter(parentID, trackID, c='blue', alpha=0.6)
plt.xlabel('Parent ID')
plt.ylabel('Track ID')
plt.title(f'Event {event_idx} Track vs Parent')
plt.grid(True)
plt.show()

# Vertex X vs Z
plt.scatter(vz, vx, c='red', alpha=0.5)
plt.xlabel('Vertex Z [mm]')
plt.ylabel('Vertex X [mm]')
plt.title('Vertex X-Z distribution')
plt.grid(True)
plt.show()

In [ ]:


# -------------------------------
# Create multi-panel figure
# -------------------------------
fig, axes = plt.subplots(2, 3, figsize=(18,10))
axes = axes.flatten()

# 1️⃣ Vertex X
axes[0].hist(vx if isinstance(vx, (list, np.ndarray)) else [vx], bins=20, color='red', alpha=0.7)
axes[0].set_xlabel('Vertex X [mm]')
axes[0].set_ylabel('Counts')
axes[0].grid(True)

# 2️⃣ Vertex Y
axes[1].hist(vy if isinstance(vy, (list, np.ndarray)) else [vy], bins=20, color='green', alpha=0.7)
axes[1].set_xlabel('Vertex Y [mm]')
axes[1].set_ylabel('Counts')
axes[1].grid(True)

# 3️⃣ Vertex Z
axes[2].hist(vz if isinstance(vz, (list, np.ndarray)) else [vz], bins=20, color='blue', alpha=0.7)
axes[2].set_xlabel('Vertex Z [mm]')
axes[2].set_ylabel('Counts')
axes[2].grid(True)

# 4️⃣ Vertex T
axes[3].hist(vt if isinstance(vt, (list, np.ndarray)) else [vt], bins=20, color='purple', alpha=0.7)
axes[3].set_xlabel('Vertex T [ns]')
axes[3].set_ylabel('Counts')
axes[3].grid(True)

# 5️⃣ Track ID vs Parent ID
axes[4].scatter(parentID, trackID, c='orange', alpha=0.6)
axes[4].set_xlabel('Parent ID')
axes[4].set_ylabel('Track ID')
axes[4].set_title('Track vs Parent')
axes[4].grid(True)

# 6️⃣ PDG histogram
axes[5].hist(PDG, bins=50, color='cyan', alpha=0.7)
axes[5].set_xlabel('PDG code')
axes[5].set_ylabel('Counts')
axes[5].grid(True)

# -------------------------------
# Figure layout
# -------------------------------
fig.suptitle(f'Event Summary — Event {event_idx}', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection='3d')

# plot each step as a line segment
for x0, y0, z0, x1, y1, z1 in zip(step_preX, step_preY, step_preZ, 
                                   step_postX, step_postY, step_postZ):
    ax.plot([x0, x1], [y0, y1], [z0, z1], color='blue')

ax.set_xlabel("X [mm]")
ax.set_ylabel("Y [mm]")
ax.set_zlabel("Z [mm]")
ax.set_title(f"Event {event_idx} Tracks")
plt.show()

In [ ]:
secStartX = tree["secStartX"].array()[event_idx]
secStartY = tree["secStartY"].array()[event_idx]
secStartZ = tree["secStartZ"].array()[event_idx]
secEndX = tree["secEndX"].array()[event_idx]
secEndY = tree["secEndY"].array()[event_idx]
secEndZ = tree["secEndZ"].array()[event_idx]

for x0, y0, z0, x1, y1, z1 in zip(secStartX, secStartY, secStartZ, 
                                   secEndX, secEndY, secEndZ):
    ax.plot([x0, x1], [y0, y1], [z0, z1], color='red', alpha=0.6)


ax.set_xlabel("X [mm]")
ax.set_ylabel("Y [mm]")
ax.set_zlabel("Z [mm]")
ax.set_title(f"Event {event_idx} Tracks")
plt.show()

In [ ]:
import uproot
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Open ROOT file
file = uproot.open("nue20gev.root")
tree = file["tree"]  # change to your actual tree name

# Choose an event
event_idx = 0

# Load primary track
step_x = np.array(tree["step_preX"].array()[event_idx].to_list())
step_y = np.array(tree["step_preY"].array()[event_idx].to_list())
step_z = np.array(tree["step_preZ"].array()[event_idx].to_list())


# Plot primary track in blue
fig = plt.figure(figsize=(10,8))
ax = fig.add_subplot(111, projection='3d')
ax.plot(step_x, step_y, step_z, color='blue', marker='', linestyle='-', alpha=0.7, label='Primary')

# Load secondaries
sec_x = tree["secStartX"].array()[event_idx]
sec_y = tree["secStartY"].array()[event_idx]
sec_z = tree["secStartZ"].array()[event_idx]

sec_ex = tree["secEndX"].array()[event_idx]
sec_ey = tree["secEndY"].array()[event_idx]
sec_ez = tree["secEndZ"].array()[event_idx]

sec_names = tree["secNames"].array()[event_idx]  # string names

# Define color mapping by particle type
color_map = {
    "gamma": "orange",
    "e-": "red",
    "e+": "green",
    "proton": "brown",
    "neutron": "grey",
    "pi+": "purple",
    "pi-": "pink",
    "pi0": "cyan",
    "mu+": "magenta",
    "mu-": "darkgreen",
}

# Plot secondaries
for sx, sy, sz, ex, ey, ez, name in zip(sec_x, sec_y, sec_z, sec_ex, sec_ey, sec_ez, sec_names):
    color = color_map.get(name, "black")  # default black if unknown
    ax.plot([sx, ex], [sy, ey], [sz, ez], color=color, alpha=0.7)

legend_elements = [
    Line2D([0], [0], color='blue', lw=2, label='Primary'),
]

# Add particle types to legend
for pname, color in color_map.items():
    legend_elements.append(
        Line2D([0], [0], color=color, lw=2, label=pname)
    )

ax.legend(handles=legend_elements, loc="upper right")

ax.set_xlabel("X [mm]")
ax.set_ylabel("Y [mm]")
ax.set_zlabel("Z [mm]")
ax.set_title(f"Event {event_idx} Display")
ax.legend()
plt.show()

In [ ]:
event_idx = 0

# Convert JaggedArrays to 1D arrays for this event
step_x = np.array(tree["step_preX"].array()[event_idx].to_list())
step_z = np.array(tree["step_preZ"].array()[event_idx].to_list())

# Plot primary track
plt.figure(figsize=(10,6))
plt.plot(step_z, step_x, color='blue', marker='o', linestyle='-', alpha=0.7, label='Primary')

# Optional: plot secondaries
sec_start_x = np.array(tree["secStartX"].array()[event_idx].to_list())
sec_start_z = np.array(tree["secStartZ"].array()[event_idx].to_list())
sec_end_x = np.array(tree["secEndX"].array()[event_idx].to_list())
sec_end_z = np.array(tree["secEndZ"].array()[event_idx].to_list())

for sx, sz, ex, ez in zip(sec_start_x, sec_start_z, sec_end_x, sec_end_z):
    plt.plot([sz, ez], [sx, ex], color='red', linestyle='--', alpha=0.6)

plt.xlabel("Z [mm] (beam direction)")
plt.ylabel("X [mm]")
plt.title(f"Event {event_idx} Display (XZ projection)")
plt.legend()
plt.grid(True)
plt.show()